# AU-saliency Probe — can Colab run it, and how long?

Throwaway benchmark. Answers **(1)** does py-feat run on a Colab T4, and **(2)** how many seconds/clip AU-on `z_v` extraction takes -> real total for the full dataset.

Colab installs **py-feat 2.x** on **Python 3.12**. The repo's `visual.py` is version-aware (handles 2.x `Detectorv1` + `detect()` and 0.6.x `Detector` + `detect_image()`), so no downgrade is needed — the old pinned torch 2.0.1 has no py3.12 wheel anyway.

**Non-destructive:** only calls `get_z_v()` (returns a tensor, writes nothing) on a few sample clips in `/content`. It never touches any feature cache.

### How to use
1. Runtime -> Change runtime type -> **T4 GPU**.
2. **Run all**. Read the verdict in the last cell.


## Cell 1 — Config

In [ ]:
N_BENCH   = 12           # clips to benchmark
AU_TOP_K  = 12           # AU runs on the top-K frames by conf x sharpness
DRIVE_ROOT        = "/content/drive/MyDrive/DeepSentinel_data"
DRIVE_SEG_ARCHIVE = DRIVE_ROOT + "/segments.zip"   # a few clips are sampled from here
REPO_URL    = "https://github.com/gjvlio/emotion-based-multimodal-deepfake-detector.git"
REPO_BRANCH = "feat/mosei-preprocess-and-eval"

# Full-dataset sizes for the extrapolation (clips needing AU-on z_v):
TOTALS = [6277, 14240, 18369]   # MOSEI-only, current cache, all-sources


## Cell 2 — Install py-feat (2.x) + visual-pipeline deps

In [ ]:
import subprocess, sys
def sh(c): print('$', c); subprocess.run(c, shell=True)
# purge any 'feat' squatter, install deps first, then real py-feat LAST
sh('pip uninstall -y feat py-feat')
sh('pip install -q transformers timm insightface onnxruntime-gpu opencv-python-headless librosa soundfile')
sh('pip install -q py-feat')
print('install done.')

## Cell 3 — Verify torch/CUDA + py-feat Detector loads

In [ ]:
import torch, feat
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
# py-feat 2.0 renamed Detector -> Detectorv1; accept either.
Detector = getattr(feat, 'Detector', None) or getattr(feat, 'Detectorv1', None)
assert Detector is not None, 'no Detector/Detectorv1 in py-feat — install failed'
print('py-feat', getattr(feat, '__version__', '?'), '-> using', Detector.__name__)
try:
    _ = Detector(au_model='xgb', device=dev)
    PYFEAT_WORKS = True
    print('py-feat OK — Detector loaded on', dev, '=> this Colab CAN run AU-on.')
except Exception as e:
    PYFEAT_WORKS = False
    import traceback; traceback.print_exc()
assert PYFEAT_WORKS, 'Detector failed to instantiate — see traceback.'

## Cell 4 — Mount Drive, clone/pull repo, grab a few sample clips

In [ ]:
from google.colab import drive
import os, zipfile
from pathlib import Path
drive.mount('/content/drive')
assert os.path.exists(DRIVE_SEG_ARCHIVE), f'segments.zip not found at {DRIVE_SEG_ARCHIVE}'
REPO_DIR = '/content/thesis'
if os.path.exists(REPO_DIR):
    subprocess.run(['git','-C',REPO_DIR,'pull'], check=True)  # get the latest version-aware visual.py
else:
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--depth','1',REPO_URL,REPO_DIR], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
SAMPLE_DIR = '/content/au_sample'; os.makedirs(SAMPLE_DIR, exist_ok=True)
with zipfile.ZipFile(DRIVE_SEG_ARCHIVE) as z:
    names = [n for n in z.namelist() if n.lower().endswith('.mp4')][:N_BENCH]
    for n in names: z.extract(n, SAMPLE_DIR)
clips = sorted(Path(SAMPLE_DIR).rglob('*.mp4'))
print(f'{len(clips)} sample clips ready; repo @ {REPO_BRANCH}')

## Cell 5 — Benchmark AU-on `z_v` (non-destructive)

Calls `get_z_v(..., AU-on)` on each clip and times it. First clip is a warm-up (model downloads + first-call overhead) and is excluded from the average.

In [ ]:
import time, statistics as st
from src.preprocessing import visual
print('py-feat API:', 'v2.x (Detectorv1)' if visual._PYFEAT_V2 else 'v0.6.x (Detector)')
visual.configure_au(enabled=True, device=dev, top_k=AU_TOP_K)
from src.preprocessing.visual import get_z_v

def extract(v):
    return get_z_v(str(v), vit_model_name='google/vit-base-patch16-224', detector='retinaface',
                   n_keyframes=8, frame_size=224, target_fps=25.0, motion_threshold=0.3,
                   confidence_threshold=0.7, device=dev)

print('warming up (ViT + insightface + py-feat first call)...')
_ = extract(clips[0])
print('warm — benchmarking', len(clips)-1, 'clips:\n')
times = []
for v in clips[1:]:
    t0 = time.time(); z = extract(v); dt = time.time() - t0; times.append(dt)
    print(f'  {v.name[:42]:<42} {dt:6.1f}s   z_v={tuple(z.shape)}')
mean = st.mean(times); med = st.median(times)
print(f'\nAU-on z_v per clip:  mean={mean:.1f}s   median={med:.1f}s   (n={len(times)})')

## Cell 6 — Verdict: does Colab work, and how long for the full run?

In [ ]:
print('='*64)
print(f'  py-feat on this Colab : WORKS  (v{getattr(__import__("feat"),"__version__","?")})')
print(f'  AU-on z_v per clip    : {med:.1f}s (median)')
print('='*64)
for total in TOTALS:
    hrs = total * med / 3600
    print(f'  {total:>6} clips  ->  {hrs:6.1f} GPU-hours  (~{hrs/24:.1f} days on ONE T4)')
    print(f'              4-way shard  ->  ~{hrs/4:.1f}h wall-clock across 4 T4s')
print()
print('z_at is REUSED (AU-independent) -> the real run rebuilds ONLY z_v.')
print('Real run (sharded on Colab OR local .venv-feat):')
print('  preprocess_all.py --use_au --au_top_k 12 \\')
print('     --out_dir data/preprocessed_au_on --reuse_zat_from data/preprocessed')
print('  (--out_dir REQUIRED with --use_au — the AU-off baseline is never overwritten.)')